In [1]:
import tensorflow as tf

In [2]:
daily_sales_numbers = [21,22,-108,31,-1,32,34,31]

In [4]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)
tf_dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [6]:
for sales in tf_dataset:
    print(sales.numpy())

21
22
-108
31
-1
32
34
31


In [8]:
for sales in tf_dataset.take(3):
    print(sales.numpy())

21
22
-108


In [11]:
tf_dataset = tf_dataset.filter(lambda x: x>0)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
31
32
34
31


In [12]:
tf_dataset = tf_dataset.map(lambda x: x*72)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1512
1584
2232
2304
2448
2232


In [15]:
tf_dataset = tf_dataset.shuffle(3)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1584
2232
2304
1512
2448
2232


In [16]:
for sales_batch in tf_dataset.batch(2):
    print(sales_batch.numpy())

[1584 1512]
[2232 2232]
[2448 2304]


In [19]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)

tf_dataset = tf_dataset.filter(lambda x: x > 0).map(lambda y: y*72).shuffle(2).batch(2)

for sales_batch in tf_dataset.batch(2):
    print(sales_batch.numpy())

[[1512 1584]
 [2304 2448]]
[[2232 2232]]


In [24]:
images_ds = tf.data.Dataset.list_files('images/*/*', shuffle=False)

for file in images_ds.take(3):
    print(file.numpy())

b'images\\cat\\20 Reasons Why Cats Make the Best Pets....jpg'
b'images\\cat\\7 Foods Your Cat Can_t Eat.jpg'
b'images\\cat\\A cat appears to have caught the....jpg'


In [25]:
images_ds = images_ds.shuffle(200)
for file in images_ds.take(3):
    print(file.numpy())

b'images\\dog\\Ancient dog DNA reveals 11_000 years of....jpg'
b'images\\cat\\7 Foods Your Cat Can_t Eat.jpg'
b'images\\dog\\Largest Dog Breeds \xe2\x80\x93 American Kennel Club.jpg'


In [26]:
class_names = ['cat', 'dog']

In [27]:
image_count = len(images_ds)
image_count

130

In [28]:
train_size = int(image_count*0.8)

train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

In [29]:
len(train_ds)

104

In [30]:
len(test_ds)

26

In [33]:
s = "images\\dog\\Ancient dog DNA reveals 11_000 years of....jpg"

s.split("\\")[-2]

'dog'

In [36]:
def get_label(file_path):
    import os
    return tf.strings.split(file_path, os.path.sep) [-2]

In [38]:
def process_image(file_path):
    label = get_label(file_path)
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img)
    img = tf.image.resize(img, [128,128])
    
    return img, label

In [35]:
for t in train_ds.take(4):
    print(t.numpy())

b'images\\dog\\List of Dog Breeds _ Petfinder.jpg'
b'images\\dog\\best dog treats_ according to veterinarians.jpg'
b'images\\dog\\Common Dog Breeds and Their Health Issues.jpg'
b'images\\dog\\Rescue turns dog with untreatable tumor....jpg'


In [48]:
train_ds = images_ds.take(train_size)
for img, label in train_ds.map(process_image).take(3):
    print("Image: ", img)
    print("Label: ", label)

Image:  tf.Tensor(
[[[129.      147.       63.     ]
  [127.      141.       62.     ]
  [132.46875 141.46875  58.46875]
  ...
  [144.      161.       81.     ]
  [145.53125 163.53125  81.53125]
  [142.9375  156.9375   79.9375 ]]

 [[127.      144.       64.     ]
  [124.59375 142.59375  60.59375]
  [130.40625 144.40625  57.40625]
  ...
  [141.      163.       81.     ]
  [143.40625 160.40625  82.40625]
  [146.      163.       83.     ]]

 [[122.34375 144.34375  62.34375]
  [125.65625 145.65625  60.65625]
  [134.65625 153.65625  61.65625]
  ...
  [145.      163.       87.     ]
  [143.      161.       85.     ]
  [144.34375 165.34375  88.34375]]

 ...

 [[ 55.34375  45.34375  44.34375]
  [ 56.03125  45.03125  43.03125]
  [ 46.3125   36.3125   34.3125 ]
  ...
  [ 98.65625 133.65625  49.65625]
  [107.      144.       51.     ]
  [104.65625 141.65625  46.65625]]

 [[ 56.40625  46.40625  47.40625]
  [ 52.59375  42.59375  41.59375]
  [ 71.40625  62.40625  63.40625]
  ...
  [114.40625 143.40

In [42]:
def scale(image, label):
    return image/255, label

In [52]:
train_ds = train_ds.map(process_image).map(scale)
for image, label in train_ds.take(5):
    print("****Image: ", image.numpy()[0][0])
    print("****Label: ", label.numpy())

****Image:  [0.46015984 0.4523167  0.29937553]
****Label:  b'dog'
****Image:  [0.49607843 0.4254902  0.38627452]
****Label:  b'cat'
****Image:  [0.843076   0.7164828  0.65545344]
****Label:  b'dog'
****Image:  [0.7567622  0.831272   0.61558574]
****Label:  b'dog'
****Image:  [0.34509805 0.19607843 0.12156863]
****Label:  b'cat'
